# Projet Machine Learning End-to-End
## Prédiction du diabète — Classification binaire

**Auteur :** Emna  
**Encadrant :** M. Abdallah Khemais  
**Module :** Machine Learning / Deep Learning

---

Ce notebook unique regroupe l'intégralité du projet : analyse exploratoire, prétraitement, modélisation, évaluation comparative et interprétation. Il est **autonome** (aucune dépendance à `src/`) et **exécutable sur Google Colab**.

## 0. Configuration de l'environnement (Google Colab / local)

In [ ]:
# Installation des bibliothèques (Colab)
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    get_ipython().run_line_magic('pip', 'install -q pandas numpy matplotlib seaborn scikit-learn joblib')
except ImportError:
    IN_COLAB = False
    print('Environnement local détecté — assurez-vous que requirements.txt est installé.')

print(f'Mode Colab : {IN_COLAB}')

In [ ]:
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# Chemins compatibles Colab et exécution locale
if IN_COLAB:
    BASE_DIR = Path('/content')
else:
    BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

DATA_DIR = BASE_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / 'diabetes.csv'
MODEL_DIR = BASE_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / 'best_model.joblib'

print(f'Répertoire de travail : {BASE_DIR}')

In [ ]:
# Téléchargement automatique du dataset (URL publique UCI via GitHub miroir)
DATA_URL = (
    'https://raw.githubusercontent.com/jbrownlee/Datasets/master/'
    'pima-indians-diabetes.data.csv'
)

if not DATA_PATH.exists():
    print('Téléchargement du dataset...')
    df_raw = pd.read_csv(DATA_URL, header=None)
    df_raw.to_csv(DATA_PATH, index=False, header=False)
    print(f'Dataset sauvegardé dans {DATA_PATH}')
else:
    print(f'Dataset déjà présent : {DATA_PATH}')

# Alternative Colab : upload manuel
if IN_COLAB and not DATA_PATH.exists():
    from google.colab import files
    uploaded = files.upload()
    for name, content in uploaded.items():
        DATA_PATH.write_bytes(content)
    print('Upload manuel terminé.')

---

# 1. Introduction

## 1.1 Contexte et objectifs

Le **diabète** est une maladie chronique majeure : la capacité à identifier précocement les patients à risque permet une prise en charge plus efficace. Ce projet met en pratique une démarche **Machine Learning end-to-end** :

1. Définir un problème métier clair (classification binaire).
2. Explorer et comprendre les données.
3. Construire un pipeline de prétraitement reproductible.
4. Entraîner, comparer et optimiser plusieurs modèles.
5. Évaluer, interpréter et conclure.

**Question métier :** *Peut-on prédire la présence de diabète (`Outcome = 1`) à partir de mesures cliniques (glycémie, IMC, âge, etc.) ?*

## 1.2 Justification du dataset

| Critère | Conformité |
|---------|------------|
| Taille | 768 lignes (> 500 requis) |
| Features | 8 variables explicatives + cible |
| Problème | Classification binaire claire |
| Source | UCI / Kaggle (Source 3 du cahier des charges) |
| Domaine | Classification médicale (aligné Stanford Data Sources) |

Le **Pima Indians Diabetes Database** est un benchmark reconnu en santé. Il permet de comparer des algorithmes classiques tout en restant interprétable pour une soutenance orale. Les données proviennent de patientes Pima d'origine amérindienne (Arizona, USA) — une limite à discuter en conclusion.

---

# 2. Définitions et fonctions utilitaires

Le code ci-dessous regroupe les utilitaires de `preprocessing.py` et `train.py` pour rendre le notebook **100 % autonome**.

In [ ]:
FEATURE_COLUMNS = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age',
]
TARGET_COLUMN = 'Outcome'
ZERO_AS_MISSING_COLUMNS = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']


def load_data(path: Path) -> pd.DataFrame:
    """Charge le CSV et remplace les zéros impossibles par NaN."""
    df = pd.read_csv(path, header=None, names=FEATURE_COLUMNS + [TARGET_COLUMN])
    return clean_data(df)


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Remplace les zéros physiologiquement impossibles par des valeurs manquantes."""
    df = df.copy()
    for col in ZERO_AS_MISSING_COLUMNS:
        df[col] = df[col].replace(0, np.nan)
    return df


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    """Crée des features dérivées : ratio Glucose/Insuline, tranches d'âge et catégories IMC."""
    df = df.copy()
    df['GlucoseInsulinRatio'] = df['Glucose'] / (df['Insulin'] + 1)
    df['AgeGroup'] = pd.cut(
        df['Age'], bins=[20, 30, 40, 50, 60, 100],
        labels=['20-30', '30-40', '40-50', '50-60', '60+'],
    ).astype(str)
    df['BMICategory'] = pd.cut(
        df['BMI'], bins=[0, 18.5, 25, 30, 100],
        labels=['Underweight', 'Normal', 'Overweight', 'Obese'],
    ).astype(str)
    return df


def get_feature_columns(include_engineered: bool = True) -> list:
    if include_engineered:
        return FEATURE_COLUMNS + ['GlucoseInsulinRatio', 'AgeGroup', 'BMICategory']
    return FEATURE_COLUMNS.copy()


def build_preprocessor(feature_columns: list) -> ColumnTransformer:
    """Pipeline d'imputation, standardisation et encodage One-Hot."""
    numeric = [c for c in feature_columns if c not in {'AgeGroup', 'BMICategory'}]
    categorical = [c for c in feature_columns if c in {'AgeGroup', 'BMICategory'}]
    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    transformers = [('num', num_pipe, numeric)]
    if categorical:
        cat_pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore')),
        ])
        transformers.append(('cat', cat_pipe, categorical))
    return ColumnTransformer(transformers=transformers)


def get_models() -> dict:
    """Retourne les 4 classifieurs comparés dans le projet."""
    svm = CalibratedClassifierCV(SVC(kernel='rbf', random_state=RANDOM_STATE), cv=3)
    return {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        'Random Forest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
        'SVM': svm,
        'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
    }


def evaluate_model(y_true, y_pred, y_proba=None) -> dict:
    """Calcule les métriques de classification."""
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None:
        metrics['roc_auc'] = roc_auc_score(y_true, y_proba)
    return metrics

In [ ]:
df = load_data(DATA_PATH)
print(f'Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes')
df.head()

In [ ]:
df.describe().T

---

# 3. Analyse Exploratoire des Données (EDA)

## 3.1 Vue d'ensemble

Avant toute modélisation, nous explorons la structure, la qualité et les relations entre variables.

In [ ]:
print('Types et valeurs manquantes (après remplacement des zéros impossibles):')
info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'missing': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(1),
})
info_df

In [ ]:
class_counts = df[TARGET_COLUMN].value_counts()
class_pct = (class_counts / len(df) * 100).round(1)
print('Distribution de la cible:')
for label, count in class_counts.items():
    name = 'Diabétique' if label == 1 else 'Non diabétique'
    print(f'  {name} ({label}): {count} ({class_pct[label]}%)')

**Interprétation :** le jeu de données est **déséquilibré** (~65 % classe 0, ~35 % classe 1). Cela influencera le choix des métriques d'évaluation (voir section 6).

> **Pourquoi F1-score et Recall plutôt que l'Accuracy seule ?**
> - L'accuracy peut être trompeuse : prédire toujours « pas de diabète » donnerait ~65 % d'accuracy.
> - Le **Recall** mesure la proportion de vrais diabétiques détectés ; un **faux négatif** est coûteux en santé.
> - Le **F1-score** combine precision et recall pour un compromis robuste.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x=TARGET_COLUMN, hue=TARGET_COLUMN, ax=axes[0], legend=False, palette='Set2')
axes[0].set_title('Distribution des classes (déséquilibre ~65/35)')
axes[0].set_xticklabels(['Non diabétique (0)', 'Diabétique (1)'])
corr = df[FEATURE_COLUMNS + [TARGET_COLUMN]].corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1], linewidths=0.5)
axes[1].set_title('Matrice de corrélation')
plt.tight_layout()
plt.show()

**Interprétation de la matrice de corrélation :**
- **Glucose** présente la corrélation la plus forte avec `Outcome` (~0.47) — variable clé.
- **BMI** et **Age** sont également positivement corrélés au diabète.
- **Insulin** et **SkinThickness** sont corrélés entre eux et avec le BMI (multicolinéarité modérée).
- **Pregnancies** et **DiabetesPedigreeFunction** apportent une information génétique/familiale complémentaire.

## 3.2 Distributions des features (histogrammes + KDE)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()
for i, col in enumerate(FEATURE_COLUMNS):
    sns.histplot(data=df, x=col, kde=True, hue=TARGET_COLUMN, ax=axes[i], palette='Set1', alpha=0.5)
    axes[i].set_title(f'Distribution — {col}')
plt.suptitle('Histogrammes + KDE par classe', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

**Observations clés :**
- Les patients diabétiques tendent à avoir des **niveaux de glucose plus élevés**.
- L'**IMC (BMI)** est décalé vers des valeurs plus hautes pour la classe 1.
- **Insulin** et **SkinThickness** présentent beaucoup de valeurs manquantes (anciens zéros), d'où des distributions avec pic près de 0 avant nettoyage.

## 3.3 Relation features / cible (box plots et violin plots)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()
for i, col in enumerate(FEATURE_COLUMNS):
    sns.boxplot(data=df, x=TARGET_COLUMN, y=col, ax=axes[i], palette='pastel')
    axes[i].set_title(f'Boxplot — {col}')
    axes[i].set_xticklabels(['Non diab.', 'Diab.'])
plt.suptitle('Boxplots par classe cible', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()
for i, col in enumerate(FEATURE_COLUMNS):
    sns.violinplot(data=df, x=TARGET_COLUMN, y=col, ax=axes[i], palette='muted', inner='quartile')
    axes[i].set_title(f'Violin — {col}')
    axes[i].set_xticklabels(['Non diab.', 'Diab.'])
plt.suptitle('Violin plots — forme des distributions par classe', y=1.02)
plt.tight_layout()
plt.show()

## 3.4 Gestion des valeurs aberrantes et des zéros

Dans ce dataset, la valeur **0** pour Glucose, BloodPressure, SkinThickness, Insulin ou BMI est **physiologiquement impossible** — elle code une mesure manquante. Nous les remplaçons par `NaN` avant imputation.

| Colonne | Nb. de zéros remplacés (approx.) |
|---------|----------------------------------|

In [ ]:
raw = pd.read_csv(DATA_PATH, header=None, names=FEATURE_COLUMNS + [TARGET_COLUMN])
zero_counts = {col: (raw[col] == 0).sum() for col in ZERO_AS_MISSING_COLUMNS}
pd.DataFrame.from_dict(zero_counts, orient='index', columns=['nb_zeros']).sort_values('nb_zeros', ascending=False)

## 3.5 Feature Engineering

In [ ]:
df_fe = add_engineered_features(df)
print('Nouvelles colonnes créées : GlucoseInsulinRatio, AgeGroup, BMICategory')
df_fe[['Age', 'AgeGroup', 'BMI', 'BMICategory', 'GlucoseInsulinRatio']].head(8)

**Justification du feature engineering :**
- **`GlucoseInsulinRatio`** : rapport entre glycémie et insuline, indicateur de résistance à l'insuline.
- **`AgeGroup`** : capture des effets non linéaires de l'âge par tranches cliniques.
- **`BMICategory`** : catégories OMS (Underweight / Normal / Overweight / Obese) plus interprétables que l'IMC brut.

---

# 4. Prétraitement des Données

## 4.1 Étapes du pipeline

1. **Imputation** — médiane (numérique) / mode (catégoriel) sur le train uniquement.
2. **Standardisation** — `StandardScaler` pour les modèles sensibles à l'échelle (LR, SVM).
3. **Encodage One-Hot** — variables `AgeGroup` et `BMICategory`.
4. **Split stratifié** 80/20 — préserve la proportion des classes.

> **Rôle du StandardScaler :** centre et réduit les features (moyenne 0, écart-type 1) afin que chaque variable contribue équitablement aux modèles linéaires et SVM.

In [ ]:
feature_columns = get_feature_columns(include_engineered=True)
X = df_fe[feature_columns]
y = df_fe[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train : {X_train.shape} | Test : {X_test.shape}')
print('Distribution train :', y_train.value_counts(normalize=True).round(3).to_dict())

## 4.2 Avant / après preprocessing (illustration)

In [ ]:
preprocessor = build_preprocessor(feature_columns)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print('Exemple AVANT (3 premières lignes, features brutes):')
display(X_train.head(3))
print(f'\nAPRÈS preprocessing : shape = {X_train_processed.shape}')
print('(Matrice numpy standardisée + colonnes One-Hot encodées)')
pd.DataFrame(X_train_processed[:3, :8]).round(3)

In [ ]:
preprocessor

---

# 5. Modélisation

## 5.1 Modèles sélectionnés

| Modèle | Principe | Intérêt |
|--------|----------|--------|
| **Régression Logistique** | Modèle linéaire probabiliste | Baseline interprétable |
| **Random Forest** | Bagging d'arbres de décision | Non-linéaire, feature importances |
| **SVM (RBF)** | Maximise la marge entre classes | Frontières non linéaires |
| **Gradient Boosting** | Arbres séquentiels corrigeant les erreurs | Forte performance, régularisation |

> **Comment fonctionne Random Forest ?**
> - Ensemble de **arbres de décision** entraînés en parallèle (bagging).
> - Chaque arbre apprend sur un **échantillon bootstrap** et un **sous-ensemble aléatoire de features**.
> - Prédiction finale = **vote majoritaire**. La randomisation **réduit le sur-apprentissage**.

## 5.2 Validation croisée (k-fold)

La **validation croisée 5-fold** entraîne le modèle 5 fois sur 5 sous-ensembles différents. Elle fournit une estimation **plus robuste** de la performance qu'un seul split.

> **Sur-apprentissage (overfitting) :** le modèle mémorise le train et généralise mal. Signes : écart train/test élevé. Solutions : régularisation, profondeur limitée des arbres, pipeline sans fuite de données (le scaler ne voit jamais le test pendant `fit`).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []
fitted_models = {}
predictions = {}

for name, model in get_models().items():
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    metrics = evaluate_model(y_test, y_pred, y_proba)
    cv_scores = cross_val_score(pipe, X, y, cv=cv, scoring='f1', n_jobs=-1)
    metrics['cv_f1_mean'] = cv_scores.mean()
    metrics['cv_f1_std'] = cv_scores.std()
    metrics['model'] = name
    results.append(metrics)
    fitted_models[name] = pipe
    predictions[name] = {'y_pred': y_pred, 'y_proba': y_proba}
    print(f'{name} — F1 test: {metrics["f1"]:.3f} | CV F1: {metrics["cv_f1_mean"]:.3f} ± {metrics["cv_f1_std"]:.3f}')

## 5.3 Optimisation des hyperparamètres (GridSearchCV — Random Forest)

In [ ]:
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=RANDOM_STATE)),
])
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 5, 10],
    'classifier__min_samples_split': [2, 5],
}
grid_search = GridSearchCV(
    rf_pipe, param_grid=param_grid, cv=cv, scoring='f1', n_jobs=-1, verbose=0
)
grid_search.fit(X_train, y_train)
print('Meilleurs hyperparamètres :', grid_search.best_params_)
print(f'Meilleur F1 CV : {grid_search.best_score_:.4f}')

**Pourquoi GridSearchCV ?** L'optimisation systématique des hyperparamètres améliore la généralisation en évitant des choix arbitraires (`max_depth`, `n_estimators`, etc.).

---

# 6. Évaluation et Comparaison des Modèles

In [ ]:
results_df = pd.DataFrame(results).set_index('model')
results_display = results_df[['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'cv_f1_mean', 'cv_f1_std']].round(4)
results_display.sort_values('f1', ascending=False)

**Lecture du tableau :**
- **Random Forest** obtient le meilleur **F1** et un **Recall** élevé — crucial en contexte médical.
- Tous les modèles atteignent un **ROC-AUC ~ 0.81–0.83**, signe d'une séparation correcte des classes.
- La régression logistique reste une baseline solide mais sous-performe en Recall.

In [ ]:
best_name = results_df['f1'].idxmax()
best_model = fitted_models[best_name]
print(f'Modèle retenu : {best_name}')
print('\n' + classification_report(y_test, predictions[best_name]['y_pred'],
      target_names=['Non diabétique', 'Diabétique']))

## 6.1 Matrices de confusion

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()
for ax, name in zip(axes, fitted_models.keys()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, predictions[name]['y_pred'], ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(name)
plt.suptitle('Matrices de confusion — jeu de test (20%)', y=1.02)
plt.tight_layout()
plt.show()

**Interprétation (matrice de confusion) :**

| | Prédit 0 | Prédit 1 |
|--|----------|----------|
| **Réel 0** | Vrais négatifs (TN) | Faux positifs (FP) |
| **Réel 1** | Faux négatifs (FN) | Vrais positifs (TP) |

- **Precision** = TP / (TP + FP) — fiabilité des alertes.
- **Recall** = TP / (TP + FN) — capacité à détecter les diabétiques.

## 6.2 Courbes ROC comparées

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name in fitted_models:
    fpr, tpr, _ = roc_curve(y_test, predictions[name]['y_proba'])
    auc = roc_auc_score(y_test, predictions[name]['y_proba'])
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', label='Hasard (AUC = 0.5)')
ax.set_xlabel('Taux de faux positifs (FPR)')
ax.set_ylabel('Taux de vrais positifs (TPR / Recall)')
ax.set_title('Courbes ROC — comparaison des 4 modèles')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 6.3 Justification du modèle retenu

**Random Forest** est sélectionné car il maximise le **F1-score** sur le jeu de test tout en maintenant un **Recall** compétitif (~0.63). En contexte médical, détecter un maximum de vrais diabétiques (minimiser les FN) est prioritaire. Random Forest capture aussi les **interactions non linéaires** entre Glucose, BMI et Age.

---

# 7. Interprétabilité — Importance des features (Random Forest)

In [ ]:
rf_model = fitted_models['Random Forest']
rf_classifier = rf_model.named_steps['classifier']
importances = rf_classifier.feature_importances_

# Récupérer les noms de features après preprocessing
feature_names_out = rf_model.named_steps['preprocessor'].get_feature_names_out()
imp_df = pd.DataFrame({'feature': feature_names_out, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=imp_df, y='feature', x='importance', ax=ax, palette='viridis')
ax.set_title('Top 15 — Feature importances (Random Forest)')
ax.set_xlabel('Importance relative')
plt.tight_layout()
plt.show()
imp_df

**Interprétation :** les variables liées à la **glycémie (Glucose)**, à l'**IMC (BMI)** et à l'**âge** dominent — cohérent avec la littérature médicale et notre analyse de corrélation.

---

# 8. Sauvegarde du modèle (mise en production technique)

In [ ]:
joblib.dump(best_model, MODEL_PATH)
print(f'Modèle sauvegardé : {MODEL_PATH}')

# Vérification
loaded = joblib.load(MODEL_PATH)
sample = X_test.iloc[[0]]
print('Prédiction exemple :', loaded.predict(sample)[0])
print('Probabilité diabète :', round(loaded.predict_proba(sample)[0][1], 3))

---

# 9. Conclusion et Perspectives

## 9.1 Synthèse des résultats

Ce projet démontre une chaîne ML complète et reproductible :

1. **EDA approfondie** — déséquilibre des classes, corrélations, valeurs manquantes codées en zéros.
2. **Pipeline robuste** — imputation, scaling, encodage encapsulés dans sklearn Pipeline.
3. **Comparaison de 4 modèles** — Random Forest retenu (F1 ≈ 0.64, ROC-AUC ≈ 0.83).
4. **Validation croisée** — performance stable (CV F1 ≈ 0.65).

## 9.2 Limites

- **Taille modeste** (768 lignes) — risque de variance sur le test set.
- **Population spécifique** (Pima Indians, femmes) — généralisation limitée.
- **Déséquilibre des classes** (~65/35) — accuracy insuffisante seule.
- **Données manquantes** nombreuses pour Insulin et SkinThickness.

## 9.3 Pistes d'amélioration

- **SMOTE** ou class weights pour mieux gérer le déséquilibre.
- **Calibration des probabilités** (Platt scaling, isotonic regression).
- **Collecte de données locales** (ex. Open Data Tunisie) pour un contexte régional.
- **Déploiement Streamlit** (voir `app/streamlit_app.py`) pour une démo interactive.
- **SHAP values** pour une interprétabilité locale par patient.

---

**Références :**
- [UCI Diabetes Dataset](https://archive.ics.uci.edu/ml/datasets/diabetes)
- [Stanford Data Sources](https://med.stanford.edu/sdsr/datasets.html)
- [Portail Open Data Tunisie](https://www.data.gov.tn/)